> **Notebook version:** staged pipeline + carbon Cα (Vydula+2024). If you still see `build_ulsa_rrl_sky` or hydrogen α picks, use **File → Revert File** then re-run all cells.

# ULSA + RRL staged analysis

Visualization for :class:`lusee.RRLAnalysis.RRLAnalysisPipeline` and `run_rrl_sim.py` output.

**Vydula et al. 2024** ([AJ 167, 2](https://doi.org/10.3847/1538-3881/ad08ba); [arXiv:2302.14185](https://arxiv.org/abs/2302.14185)):

| Component | Model |
|-----------|--------|
| Smooth envelope on catalog `(l,b)` | Eq. (7) \(T_{\mathrm{RRL}}(\nu)\); `RRL_GAS_CASE` `cold` (Cα) or `hot` (Hα) |
| Discrete lines after resampling | Eq. (1)–(2) **carbon Cα**; all transitions in the analysis band |

**Stages:** ULSA + envelope → Croissant @ beam Δν → resample (`FINE_STEP_KHZ`) → add Cα spectrum.

## Run order

1. Imports → 2. Setup → 3. Mollweide maps → 4. Injected Cα lines → 5. FITS catalog *(optional)* → 6. `pipeline.run` *(croissant)* → 7. Waterfall FITS

Set `LUSEE_DRIVE_DIR`. Driver: `simulation/config/rrl_fine_1khz.yaml`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt
import fitsio
import healpy as hp

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import lusee as lu


In [ ]:
from lusee.frequencies import fine_uniform_frequency_mhz

drive = os.environ.get("LUSEE_DRIVE_DIR")
ulsa_file = (
    os.path.join(drive, "Simulations", "SkyModels", "ULSA_32_ddi_smooth.fits")
    if drive
    else ""
)
rrl_file = (
    os.path.join(
        drive,
        "Simulations",
        "SkyModels",
        "RRL_maps",
        "RRL_H166-167-168a_HIPASS+ZOA_lbv.fits",
    )
    if drive
    else ""
)

use_drive = bool(drive and os.path.isfile(ulsa_file) and os.path.isfile(rrl_file))

RRL_SPOT_SIGMA_DEG = 0.35
RRL_GAS_CASE = "cold"  # Eq. (7): cold (Cα) or hot (Hα)
RRL_LINE_SIGMA_MHZ = lu.RRL_DEFAULT_LINE_SIGMA_MHZ
RRL_LINE_PEAK_K = lu.RRL_DEFAULT_LINE_PEAK_K
FINE_STEP_KHZ = 0.5

FREQ_BEAM_MHZ = fine_uniform_frequency_mhz(10.0, 15.0, step_khz=50.0)
NU_LO = float(FREQ_BEAM_MHZ[0])
NU_HI = float(FREQ_BEAM_MHZ[-1])

if use_drive:
    lmax = 48
    ulsa_path = ulsa_file
    rrl_cat_path = rrl_file
    print("Drive data:", ulsa_path)
else:
    root = tempfile.mkdtemp(prefix="rrl_nb_")
    nside, lmax = 8, 3 * 8 - 1
    npix = 12 * nside**2
    f0, f1, df = 10.0, 15.0, 1.0
    nfreq = int(round((f1 - f0) / df)) + 1
    maps = np.full((nfreq, npix), 55.0, dtype=np.float32)
    ulsa_path = os.path.join(root, "ulsa.fits")
    fitsio.write(
        ulsa_path,
        maps,
        header={"freq_start": f0, "freq_end": f1, "freq_step": df},
    )
    rrl_cat_path = os.path.join(root, "rrl.fits")
    fitsio.write(
        rrl_cat_path,
        {"GLON": np.array([120.0]), "GLAT": np.array([10.0])},
        extname="SRC",
    )
    print("Synthetic FITS in", root)

pipeline = lu.build_rrl_analysis_pipeline(
    lmax,
    ulsa_path=ulsa_path,
    rrl_catalog_path=rrl_cat_path,
    lusee_drive_dir=drive if use_drive else None,
    fine_step_khz=FINE_STEP_KHZ,
    spot_sigma_deg=RRL_SPOT_SIGMA_DEG,
    gas_case=RRL_GAS_CASE,
    rrl_sigma_mhz=RRL_LINE_SIGMA_MHZ,
    rrl_peak_k=RRL_LINE_PEAK_K,
)

N_CALPHA = len(lu.carbon_rrl_alpha_transitions_in_frequency_band_mhz(NU_LO, NU_HI))
print(
    f"Pipeline ready: beam {len(FREQ_BEAM_MHZ)} ch, fine Δν={FINE_STEP_KHZ} kHz, "
    f"gas_case={RRL_GAS_CASE!r}, Cα lines in band N={N_CALPHA}"
)


In [ ]:
# Mollweide maps: ULSA, envelope (Eq. 7), spot template, ULSA+envelope — no Croissant
sky_env = pipeline.build_envelope_sky(FREQ_BEAM_MHZ)

nu_pick = 0.5 * (NU_LO + NU_HI)
fi = int(np.argmin(np.abs(np.asarray(sky_env.freq, dtype=float) - nu_pick)))
nu_eff = float(np.asarray(sky_env.freq)[fi])

nu_arr = np.asarray([nu_eff], dtype=np.float64)
if sky_env.gas_case == "gaussian":
    t_rrl_nu = float(
        lu.rrl_smooth_envelope_weight_mhz(
            nu_arr,
            nu_ref_mhz=sky_env.envelope_nu_ref_mhz,
            sigma_mhz=sky_env.envelope_sigma_mhz,
            amplitude_k=sky_env.envelope_amplitude_k,
            gas_case="gaussian",
        )[0]
    )
else:
    t_rrl_nu = float(
        lu.rrl_envelope_T_rrl_k_mhz(
            nu_arr, sky_env.envelope_params, gas_case=sky_env.gas_case
        )[0]
    )

m_env_K = sky_env.envelope_brightness_map_K(fi)
hp.mollview(
    m_env_K,
    coord="G",
    title=(
        f"RRL envelope @ {nu_eff:.4f} MHz; gas_case={sky_env.gas_case!r}\n"
        f"T_RRL(ν)≈{t_rrl_nu:.4f} K × spot alm"
    ),
    unit="K",
    cmap="magma",
)
plt.show()

hp.mollview(
    sky_env.rrl_spot_map,
    coord="G",
    title="Catalog Gaussian spots (before alm band-limit)",
    cmap="magma",
)
plt.show()

alm_ulsa = np.asarray(sky_env.mapalm_ulsa[fi], dtype=np.complex128)
m_ulsa_K = np.real(
    hp.alm2map(alm_ulsa, nside=int(sky_env.Nside), lmax=int(sky_env.lmax))
).astype(np.float64)
hp.mollview(
    m_ulsa_K,
    coord="G",
    title=f"ULSA @ {nu_eff:.4f} MHz",
    unit="K",
    cmap="magma",
)
plt.show()

alm_total = np.asarray(sky_env.get_alm(np.asarray([fi]))[0], dtype=np.complex128)
m_ulsa_env_K = np.real(
    hp.alm2map(alm_total, nside=int(sky_env.Nside), lmax=int(sky_env.lmax))
).astype(np.float64)
hp.mollview(
    m_ulsa_env_K,
    coord="G",
    title=f"ULSA + envelope @ {nu_eff:.4f} MHz (Croissant sky input)",
    unit="K",
    cmap="magma",
)
plt.show()


In [ ]:
# Stage 5 preview: carbon Cα lines (Vydula+2024 Eq. 1–2) on the fine grid
freq_fine_plot = pipeline.fine_frequency_mhz(NU_LO, NU_HI)
transitions = lu.carbon_rrl_alpha_transitions_in_frequency_band_mhz(NU_LO, NU_HI)

line_k = lu.rydberg_line_spectrum_mhz(
    freq_fine_plot,
    None,
    nu_lo_mhz=NU_LO,
    nu_hi_mhz=NU_HI,
    species="carbon",
    sigma_mhz=pipeline.rrl_sigma_mhz,
    peak_k=pipeline.rrl_peak_k,
)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(freq_fine_plot, line_k, color="C1", lw=0.7, label=f"Σ Cα (N={len(transitions)})")
step = max(1, len(transitions) // 8)
for (n1, n2) in transitions[::step]:
    nu0 = lu.carbon_rrl_frequency_mhz(n1, n2)
    ax.axvline(nu0, color="C1", ls="--", lw=0.5, alpha=0.4)
    ax.text(nu0, pipeline.rrl_peak_k * 1.02, f"C{n1}α", ha="center", va="bottom", fontsize=7)
ax.set_xlim(NU_LO, NU_HI)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("ΔT [K]")
ax.set_title(
    f"Injected carbon Cα (σ={pipeline.rrl_sigma_mhz * 1e3:.2f} kHz, peak={pipeline.rrl_peak_k} K)"
)
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Cα in [{NU_LO}, {NU_HI}] MHz: N={len(transitions)}")
print(f"  first {transitions[0]} → {lu.carbon_rrl_frequency_mhz(*transitions[0]):.6f} MHz")
print(f"  last  {transitions[-1]} → {lu.carbon_rrl_frequency_mhz(*transitions[-1]):.6f} MHz")
print(f"Fine grid: {freq_fine_plot.size} ch, Δν={FINE_STEP_KHZ} kHz")


In [ ]:
# Optional: RRL FITS catalog positions
from astropy.io import fits as afits

path = rrl_cat_path
print("RRL path:", path)
with afits.open(path, memmap=True) as hdul:
    for i, hdu in enumerate(hdul):
        d = hdu.data
        shape = getattr(d, "shape", None)
        print(f"  HDU {i} name={hdu.name!r} kind={type(hdu).__name__} shape={shape}")

lon, lat = lu.load_rrl_region_positions_gal_deg(path)
print(f"\nCatalog positions: N = {len(lon)} (l, b) pairs")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(lon, lat, s=1, alpha=0.4)
ax.set_xlabel("Galactic longitude (deg)")
ax.set_ylabel("Galactic latitude (deg)")
ax.set_title("RRL catalog (Gaussian centres)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Run staged pipeline (needs croissant + s2fft)
from lusee.beam_fine_frequency import linear_resample_beam_freq_mhz

if lu.CroSimulator is None:
    raise RuntimeError("Install croissant-sim and s2fft for the staged pipeline.")

beam0 = lu.BeamGauss(
    alt_deg=90.0,
    az_deg=0.0,
    sigma_deg=25.0,
    one_over_freq_scaling=False,
    id="nb_rrl",
)
beam = linear_resample_beam_freq_mhz(beam0, FREQ_BEAM_MHZ)

obs = lu.Observation(
    "2025-03-01 00:00:00 to 2025-03-01 06:00:00",
    deltaT_sec=3600.0,
    lun_lat_deg=0.0,
    lun_long_deg=0.0,
)

st = pipeline.run(obs, beam, CroSimulator=lu.CroSimulator, Tground=0.0, combinations=[(0, 0)])
freq_beam = st.freq_beam_mhz
freq_fine = st.freq_fine_mhz
print(f"Ran pipeline: {len(freq_beam)} beam ch → {len(freq_fine)} fine ch, Cα lines injected")

fi = len(freq_beam) // 2
sky_env = st.envelope_sky
m_ulsa = np.real(
    hp.alm2map(
        np.asarray(sky_env.mapalm_ulsa[fi], dtype=np.complex128),
        nside=sky_env.Nside,
        lmax=sky_env.lmax,
    )
)
m_env = sky_env.envelope_brightness_map_K(fi)
for m, title in (
    (m_ulsa, "After run: ULSA"),
    (m_env, "After run: envelope"),
    (m_ulsa + m_env, "After run: ULSA + envelope"),
):
    hp.mollview(m, coord="G", title=title, unit="K", cmap="magma")
    plt.show()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(freq_beam, st.waterfall_beam[0, 0, :], "k-", lw=0.8)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("T [K]")
ax.set_title("Stage 3: Croissant (beam resolution)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(freq_fine, st.waterfall_resampled[0, 0, :], color="C0", lw=0.6)
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("T [K]")
ax.set_title(f"Stage 4: resampled (Δν={FINE_STEP_KHZ} kHz)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(freq_fine, st.line_spectrum_fine_k, "C1", lw=0.8, label="carbon Cα only")
ax.plot(freq_fine, st.waterfall_final[0, 0, :], "k-", lw=0.5, alpha=0.8, label="final")
ax.set_xlabel("Frequency (MHz)")
ax.set_ylabel("T [K]")
ax.set_title("Stage 5: + carbon Cα → final")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Simulator waterfall (FITS)

```bash
export LUSEE_DRIVE_DIR=... LUSEE_OUTPUT_DIR=...
python simulation/driver/run_rrl_sim.py simulation/config/rrl_fine_1khz.yaml
```

Loads `rrl_sim_waterfall.fits` (same pipeline; `gas_case: cold` and carbon Cα in YAML band).


In [ ]:
_waterfall_path = os.environ.get("RRL_WATERFALL_FITS")
if _waterfall_path and os.path.isfile(_waterfall_path):
    waterfall_fits = _waterfall_path
else:
    _out = os.environ.get("LUSEE_OUTPUT_DIR", ".")
    waterfall_fits = os.path.join(_out, "rrl_sim_waterfall.fits")

wf_data = wf_freq = None
if os.path.isfile(waterfall_fits):
    with fitsio.FITS(waterfall_fits) as fits:
        wf_data = fits["data"].read()
        wf_freq = np.asarray(fits["freq"].read(), dtype=float).reshape(-1)
    print("Loaded", waterfall_fits, "shape", wf_data.shape)
else:
    print("No waterfall at", waterfall_fits, "— run run_rrl_sim.py first")


In [ ]:
if wf_data is None:
    print("Skipping waterfall plot.")
else:
    _nt, _nc, _nf = wf_data.shape
    _lo, _hi = np.nanpercentile(wf_data[:, 0, :], [2, 98])
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(
        wf_data[:, 0, :],
        aspect="auto",
        origin="lower",
        extent=[float(wf_freq[0]), float(wf_freq[-1]), 0, _nt - 1],
        vmin=_lo,
        vmax=_hi,
        cmap="magma",
    )
    ax.set_xlabel("Frequency (MHz)")
    ax.set_ylabel("Time index")
    ax.set_title("Driver waterfall (staged pipeline + carbon Cα)")
    plt.colorbar(im, ax=ax, label="T [K]")
    plt.tight_layout()
    plt.show()
